# Choosing a reasoning effort for accuracy, latency, and cost

Reasoning models such as `o3-mini` generate hidden *reasoning tokens* before they write a final answer. The `reasoning_effort` parameter (`"low"`, `"medium"`, or `"high"`) controls how much internal reasoning the model is allowed to do; `medium` is the default. Higher effort can improve answers on difficult problems, but it also increases latency, token usage, and cost.

There is no universally best setting: the right choice depends on the task. In this notebook we build a small, repeatable evaluation that measures accuracy, latency, token usage, and cost at each effort level. We then use the results to pick a default and to identify where extra reasoning is worth paying for.

The same harness can be pointed at a sample of your own workload; replace the four tasks below with tasks that represent what your application actually does.

## 1. Setup

Install the dependencies and create an API client. The notebook reads `OPENAI_API_KEY` from the environment. The prices below are for `o3-mini`; update `MODEL` and the prices if you use another reasoning model.

In [ ]:
%pip install --upgrade openai pandas matplotlib

In [ ]:
import re
import time

import matplotlib.pyplot as plt
import pandas as pd
from openai import OpenAI

client = OpenAI()

MODEL = "o3-mini"  # Any reasoning model that supports reasoning_effort
EFFORTS = ["low", "medium", "high"]
MAX_COMPLETION_TOKENS = 4096

# o3-mini pricing in USD per 1M tokens (February 2025).
# Reasoning tokens are billed as output tokens. Check the pricing page for current values:
# https://openai.com/api/pricing/
INPUT_PRICE_PER_MILLION = 1.10
OUTPUT_PRICE_PER_MILLION = 4.40

## 2. Define a small evaluation set

Each task has a prompt with a single verifiable answer. The prompts ask the model to end with a `Final answer:` line so that responses can be scored automatically.

These tasks are intentionally short. They are a smoke test for the measurement workflow, not a benchmark: four tasks are enough to see the shape of the accuracy/cost tradeoff, but production decisions should use a larger, representative evaluation set.

In [ ]:
TASKS = [
    {
        "id": "word_problem",
        "prompt": (
            "A store sells pens in packs of 7. Maya buys 5 packs and gives 9 pens away. "
            "How many pens does she have left?"
        ),
        "expected": "26",
    },
    {
        "id": "sequence",
        "prompt": "What number comes next in the sequence 2, 6, 12, 20, 30, ...?",
        "expected": "42",
    },
    {
        "id": "logic",
        "prompt": (
            "Three friends, Ada, Ben, and Cy, each have a different pet: a cat, a dog, "
            "and a bird. Ada does not have the cat. Ben has the dog. Who has the bird?"
        ),
        "expected": "Ada",
    },
    {
        "id": "code",
        "prompt": (
            "What does this Python code print?\n\n"
            "values = [i * i for i in range(5)]\n"
            "print(sum(values))"
        ),
        "expected": "30",
    },
]

## 3. Parse answers and call the model

The response text is parsed for the last `Final answer:` line, normalized (case and punctuation are ignored), and compared with the expected value.

Each call also records:

- `reasoning_tokens` from `usage.completion_tokens_details.reasoning_tokens`, which shows how many hidden tokens the model spent thinking;
- total `completion_tokens` and `prompt_tokens`;
- wall-clock latency;
- an estimated cost using the per-million-token prices above.

In [ ]:
FINAL_ANSWER_PATTERN = re.compile(r"final answer\s*:\s*(.+)", re.IGNORECASE)


def extract_final_answer(text: str) -> str:
    """Return the last value after a 'Final answer:' line, without surrounding punctuation."""
    matches = FINAL_ANSWER_PATTERN.findall(text or "")
    if not matches:
        return ""
    answer = matches[-1].strip()
    return answer.strip(" \t`*_.").rstrip(".")


def normalize(answer: str) -> str:
    """Normalize an answer for exact-match scoring."""
    return re.sub(r"[^a-z0-9]+", "", answer.lower())

In [ ]:
def run_task(task: dict, effort: str) -> dict:
    """Run one task at one reasoning effort and collect quality and cost signals."""
    started = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "developer",
                "content": (
                    "You are a careful problem solver. Work through the problem, then end "
                    "with a line exactly in the form: Final answer: <value>. The value must "
                    "be only the requested number or name."
                ),
            },
            {"role": "user", "content": task["prompt"]},
        ],
        reasoning_effort=effort,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )
    latency_s = time.perf_counter() - started

    content = response.choices[0].message.content or ""
    usage = response.usage
    reasoning_tokens = getattr(
        getattr(usage, "completion_tokens_details", None), "reasoning_tokens", 0
    ) or 0
    answer = extract_final_answer(content)

    return {
        "task": task["id"],
        "effort": effort,
        "expected": task["expected"],
        "answer": answer,
        "correct": normalize(answer) == normalize(task["expected"]),
        "finish_reason": response.choices[0].finish_reason,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "reasoning_tokens": reasoning_tokens,
        "latency_s": latency_s,
        "cost_usd": (
            usage.prompt_tokens * INPUT_PRICE_PER_MILLION
            + usage.completion_tokens * OUTPUT_PRICE_PER_MILLION
        )
        / 1_000_000,
        "response": content,
    }

## 4. Run every task at every effort level

This makes 4 tasks × 3 effort levels = 12 requests. Lower `MAX_COMPLETION_TOKENS` or use fewer tasks while experimenting if you want to reduce cost.

In [ ]:
results = []
for task in TASKS:
    for effort in EFFORTS:
        print(f"Running {task['id']} at {effort} effort...")
        results.append(run_task(task, effort))

df = pd.DataFrame(results)
df[["task", "effort", "correct", "reasoning_tokens", "completion_tokens", "latency_s", "cost_usd"]]

## 5. Summarize the tradeoff

Grouping by effort shows the average accuracy, reasoning-token usage, latency, and cost per request. Cost per 1,000 requests makes the difference easier to reason about.

In [ ]:
summary = (
    df.groupby("effort")
    .agg(
        accuracy=("correct", "mean"),
        mean_reasoning_tokens=("reasoning_tokens", "mean"),
        mean_completion_tokens=("completion_tokens", "mean"),
        mean_latency_s=("latency_s", "mean"),
        mean_cost_usd=("cost_usd", "mean"),
    )
    .reindex(EFFORTS)
)
summary["cost_per_1k_requests_usd"] = summary["mean_cost_usd"] * 1000
summary

## 6. Visualize accuracy, latency, and cost

Plotting the three dimensions together makes it clear when higher effort buys accuracy and when it only adds cost.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

summary["accuracy"].plot.bar(ax=axes[0], color="#10a37f")
axes[0].set_title("Accuracy")
axes[0].set_ylabel("Fraction correct")
axes[0].set_ylim(0, 1.05)

summary["mean_latency_s"].plot.bar(ax=axes[1], color="#10a37f")
axes[1].set_title("Latency")
axes[1].set_ylabel("Seconds per request")

summary["cost_per_1k_requests_usd"].plot.bar(ax=axes[2], color="#10a37f")
axes[2].set_title("Estimated cost")
axes[2].set_ylabel("USD per 1,000 requests")

for ax in axes:
    ax.set_xlabel("Reasoning effort")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 7. Inspect per-task results

A single aggregate number can hide an important pattern: extra effort often helps the hardest task while changing nothing on the easy ones. The per-task table shows where the gains come from.

In [ ]:
df.pivot_table(
    index="task", columns="effort", values="correct", aggfunc="first"
).reindex(columns=EFFORTS)

## Interpreting the results

Use the lowest effort that clears your accuracy bar:

- **Start at `low`.** For classification, extraction, routine rewriting, and other well-specified tasks, the extra reasoning at `medium` or `high` often changes nothing.
- **Escalate only where the evaluation shows failures.** If `high` fixes specific hard tasks, route those tasks to `high` and keep the rest cheap.
- **Watch reasoning tokens, not just visible output.** `completion_tokens` includes reasoning tokens, so a short final answer can still be expensive. The `reasoning_tokens` column shows where the output budget goes.
- **Treat small evaluations with caution.** Four tasks and one run per condition are noisy. Before locking in a default, use a sample of real requests (often 50–200), and consider repeating each condition to average out variance.
- **Check the finish reason.** `max_completion_tokens` limits reasoning and visible output together. If `finish_reason` is `"length"`, the model was cut off before writing an answer, so raise the limit.
- **Re-run when anything changes.** Model updates, prompt changes, and new tool definitions can all move the accuracy/cost curve.

Effort is only one lever. If `high` is still not accurate enough, the next step is usually a better prompt, a stronger model, or decomposing the task into smaller steps rather than increasing effort further.

### Next steps

- Replace `TASKS` with a labeled sample from your application.
- Add a target metric other than exact match (for example, a rubric or an LLM judge) for open-ended outputs.
- Cache or batch requests if you are running a large evaluation, and review the [pricing page](https://openai.com/api/pricing/) before using these estimates in a budget.